In [23]:
from pathlib import Path
import json
import re
import sys
import os
import inspect
from typing import Any, Dict, List, Optional

import torch
import pandas as pd
from tqdm.auto import tqdm
from datasets import load_dataset, get_dataset_config_names
from transformers import AutoTokenizer, AutoModelForCausalLM

import os
from huggingface_hub import login, whoami
from getpass import getpass

from evaluate.extract_answer import extract_boxed_answer, is_equiv

In [26]:
os.environ["HF_TOKEN"] = getpass("Paste your HF token: ")
login(token=os.environ["HF_TOKEN"])

whoami(token=os.environ["HF_TOKEN"])

Paste your HF token:  ········


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


{'type': 'user',
 'id': '666b03c6d5e7ca261c77c468',
 'name': 'emma-anastassova',
 'fullname': 'Emma Anastassova',
 'isPro': False,
 'avatarUrl': '/avatars/1d2e123afae29b0650ead4c462dd8a00.svg',
 'orgs': [{'type': 'org',
   'id': '69f9bb793215252d1c045bcc',
   'name': 'mystery-machine',
   'fullname': 'Mystery Machine',
   'email': 'hugo.demulerey@gmail.com',
   'canPay': False,
   'billingMode': 'prepaid',
   'periodEnd': 1780272000,
   'avatarUrl': 'https://cdn-avatars.huggingface.co/v1/production/uploads/69efbeea1eee727f224ffea0/HSXYTmoMqSLuiimzrxysQ.webp',
   'roleInOrg': 'write'},
  {'type': 'org',
   'id': '69fb87d3a5fe2c0acc1cba16',
   'name': 'cs-552-2026-mystery-machine',
   'fullname': 'Mystery Machine',
   'email': 'emma.anastassova@gmail.com',
   'canPay': False,
   'billingMode': 'prepaid',
   'periodEnd': 1780272000,
   'avatarUrl': 'https://cdn-avatars.huggingface.co/v1/production/uploads/666b03c6d5e7ca261c77c468/92GKgofkeN67eY6Wz5qGU.webp',
   'roleInOrg': 'admin'}],
 'a

In [49]:
REPO = Path("/scratch/workspace/emma/standard-project-m2-mystery-machine")
sys.path.append(str(REPO))

RESULTS_DIR = REPO / "multilingual" / "results" / "base_qwen"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "Qwen/Qwen3-1.7B"

TARGET_LANGUAGES = {
    "russian": {
        "include_config": "Russian",
        "mgsm_configs": ["ru", "russian"],
    },
    "chinese": {
        "include_config": "Chinese",
        "mgsm_configs": ["zh", "chinese", "zh_cn"],
    },
    "italian": {
        "include_config": "Italian",
        "mgsm_configs": ["it", "italian"],
    },
    "spanish": {
        "include_config": "Spanish",
        "mgsm_configs": ["es", "spanish"],
    },
    "hindi": {
        "include_config": "Hindi",
        "mgsm_configs": ["hi", "hindi"],
    },
    "english": {
        "include_config": None,
        "mgsm_configs": ["en", "english"],
    },
}

N_PER_LANGUAGE = 20 # just to ttest
MAX_NEW_TOKENS_INCLUDE = 128
MAX_NEW_TOKENS_MGSM = 512
MAX_NEW_TOKENS_MC = 128
MAX_NEW_TOKENS_MATH = 512

LETTERS = [chr(ord("A") + i) for i in range(26)]

In [35]:
hf_token = os.environ.get("HF_TOKEN")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=hf_token,
    trust_remote_code=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    token=hf_token,
    dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

model.eval()

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 2048)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=2048, out_features=6144, bias=False)
          (up_proj): Linear(in_features=2048, out_features=6144, bias=False)
          (down_proj): Linear(in_features=6144, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((2048,), eps=1e-06)
        (post_attention_layer

In [36]:
# generation
def build_messages(prompt: str, task_type: str) -> List[Dict[str, str]]:
    if task_type == "mcq":
        system = (
            "You are a careful multilingual question-answering assistant. "
            "Answer the multiple-choice question. "
            "Put only the final option letter inside \\boxed{...}. /no_think"
        )
    elif task_type == "math":
        system = (
            "You are a careful multilingual mathematical reasoning assistant. "
            "Solve the problem. "
            "Put only the final numeric answer inside \\boxed{...}. /no_think"
        )
    else:
        system = (
            "You are a careful multilingual assistant. "
            "Put the final answer inside \\boxed{...}. /no_think"
        )

    return [
        {"role": "system", "content": system},
        {"role": "user", "content": prompt},
    ]

@torch.inference_mode()
def generate_answer(prompt: str, task_type: str, max_new_tokens: int) -> str:
    messages = build_messages(prompt, task_type)

    chat_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(chat_text, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

    generated = outputs[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(generated, skip_special_tokens=False)

In [37]:
# helpers
def pick_split(ds) -> str:
    for split in ["test", "validation", "dev", "train"]:
        if split in ds:
            return split
    return list(ds.keys())[0]


def normalize_mc_answer(x: Any) -> Optional[str]:
    if x is None:
        return None
    return str(x).strip().upper()


def normalize_text_answer(x: Any) -> Optional[str]:
    if x is None:
        return None
    return str(x).strip()


def select_at_most(dataset, n=None):
    dataset = dataset.shuffle(seed=42)
    if n is None:
        return dataset
    n_eff = min(n, len(dataset))
    return dataset.select(range(n_eff))


def save_jsonl(df: pd.DataFrame, dataset_name: str, language: str) -> Path:
    out_path = RESULTS_DIR / f"{dataset_name}_{language}_base_qwen.jsonl"
    df.to_json(out_path, orient="records", lines=True, force_ascii=False)
    print(f"Saved: {out_path}")
    return out_path


### INCLUDE formatting

In [46]:
def detect_include_language_field(dataset) -> str:
    for candidate in ["language", "lang", "locale"]:
        if candidate in dataset.features:
            return candidate
    raise KeyError(f"No language field found. Features: {dataset.features}")


def format_include_example(ex: Dict[str, Any], language: str, idx: int) -> Dict[str, Any]:
    question = str(ex["question"])

    choices = [
        str(ex["option_a"]),
        str(ex["option_b"]),
        str(ex["option_c"]),
        str(ex["option_d"]),
    ]

    options = "\n".join(
        f"{LETTERS[i]}) {choice}"
        for i, choice in enumerate(choices)
    )

    prompt = (
        f"{question}\n\n"
        f"{options}\n\n"
        "Answer with the correct option letter inside \\boxed{...}."
    )

    gold = LETTERS[int(ex["answer"])]

    metadata = {
        "language": ex["language"],
        "country": ex["country"],
        "domain": ex["domain"],
        "subject": ex["subject"],
        "regional_feature": ex["regional_feature"],
        "level": ex["level"],
    }

    return {
        "id": f"include_{language}_{idx}",
        "dataset": "include",
        "language": language,
        "task_type": "mcq",
        "prompt": prompt,
        "answer": gold,
        "metadata": metadata,
    }


def load_include_examples(language: str, n: int) -> List[Dict[str, Any]]:
    include_config = TARGET_LANGUAGES[language]["include_config"]

    if include_config is None:
        print(f"[INCLUDE] No INCLUDE config for {language}; skipping.")
        return []

    ds = load_dataset("CohereLabs/include-base-44", include_config)

    # Use test for baseline evaluation, not validation.
    split = "test" if "test" in ds else pick_split(ds)
    data = ds[split]

    if len(data) == 0:
        print(f"[INCLUDE] Empty split for {language}.")
        return []

    sampled = select_at_most(data, n)

    return [
        format_include_example(sampled[i], language, i)
        for i in range(len(sampled))
    ]

### MGSM formatting

In [39]:
def infer_mgsm_question_key(ex: Dict[str, Any]) -> str:
    for key in ["question", "problem", "input", "query", "prompt"]:
        if key in ex:
            return key
    raise KeyError(f"Could not infer MGSM question key. Keys: {list(ex.keys())}")


def infer_mgsm_answer_key(ex: Dict[str, Any]) -> str:
    for key in ["answer", "target", "output", "label"]:
        if key in ex:
            return key
    raise KeyError(f"Could not infer MGSM answer key. Keys: {list(ex.keys())}")


def format_mgsm_example(ex: Dict[str, Any], language: str, idx: int) -> Dict[str, Any]:
    question_key = infer_mgsm_question_key(ex)
    answer_key = infer_mgsm_answer_key(ex)

    question = str(ex[question_key])
    gold = str(ex[answer_key]).strip()

    prompt = (
        f"{question}\n\n"
        "Solve the problem. Put only the final numeric answer inside \\boxed{...}."
    )

    metadata = {
        k: ex[k]
        for k in ex.keys()
        if k not in {question_key, answer_key}
    }

    return {
        "id": f"mgsm_{language}_{idx}",
        "dataset": "mgsm",
        "language": language,
        "task_type": "math",
        "prompt": prompt,
        "answer": gold,
        "metadata": metadata,
    }


def load_mgsm_examples(language: str, n: int) -> List[Dict[str, Any]]:
    available_configs = get_dataset_config_names("juletxara/mgsm")
    config_lookup = {c.lower(): c for c in available_configs}

    chosen_config = None
    for candidate in TARGET_LANGUAGES[language]["mgsm_configs"]:
        if candidate.lower() in config_lookup:
            chosen_config = config_lookup[candidate.lower()]
            break

    if chosen_config is None:
        print(f"[MGSM] No config found for {language}. Available configs: {available_configs}")
        return []

    ds = load_dataset("juletxara/mgsm", chosen_config)
    split = pick_split(ds)
    data = ds[split]

    if len(data) == 0:
        print(f"[MGSM] Empty split for {language}.")
        return []

    sampled = select_at_most(data, n)

    return [
        format_mgsm_example(sampled[i], language, i)
        for i in range(len(sampled))
    ]

# Evaluation

In [47]:
def score_prediction(prediction: Optional[str], gold: str, task_type: str) -> bool:
    if task_type == "mcq":
        return normalize_mc_answer(prediction) == normalize_mc_answer(gold)

    if task_type == "math":
        return is_equiv(prediction, gold)

    return normalize_text_answer(prediction) == normalize_text_answer(gold)


def evaluate_examples(
    examples: List[Dict[str, Any]],
    dataset_name: str,
    language: str,
) -> pd.DataFrame:
    rows = []

    for ex in tqdm(examples, desc=f"{dataset_name}/{language}"):
        max_new_tokens = (
            MAX_NEW_TOKENS_MGSM
            if ex["task_type"] == "math"
            else MAX_NEW_TOKENS_INCLUDE
        )

        output = generate_answer(
            prompt=ex["prompt"],
            task_type=ex["task_type"],
            max_new_tokens=max_new_tokens,
        )

        prediction = extract_boxed_answer(output)
        correct = score_prediction(prediction, ex["answer"], ex["task_type"])

        rows.append({
            "id": ex["id"],
            "dataset": dataset_name,
            "language": language,
            "task_type": ex["task_type"],
            "gold": ex["answer"],
            "prediction": prediction,
            "correct": correct,
            "prompt": ex["prompt"],
            "output": output,
            "metadata": json.dumps(ex.get("metadata", {}), ensure_ascii=False),
        })

    df = pd.DataFrame(rows)

    if len(df) > 0:
        n_correct = int(df["correct"].sum())
        n_total = len(df)
        acc = n_correct / n_total
        print(f"{dataset_name}/{language}: accuracy = {acc:.3f} ({n_correct}/{n_total})")
    else:
        print(f"{dataset_name}/{language}: no examples")

    return df

In [50]:
all_results = []

for language in TARGET_LANGUAGES:
    include_examples = load_include_examples(language, N_PER_LANGUAGE)

    if len(include_examples) > 0:
        df_include = evaluate_examples(
            include_examples,
            dataset_name="include",
            language=language,
        )
        save_jsonl(df_include, "include", language)
        all_results.append(df_include)

    mgsm_examples = load_mgsm_examples(language, N_PER_LANGUAGE)

    if len(mgsm_examples) > 0:
        df_mgsm = evaluate_examples(
            mgsm_examples,
            dataset_name="mgsm",
            language=language,
        )
        save_jsonl(df_mgsm, "mgsm", language)
        all_results.append(df_mgsm)

include/russian:   0%|          | 0/20 [00:00<?, ?it/s]

include/russian: accuracy = 0.350 (7/20)
Saved: /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/results/base_qwen/include_russian_base_qwen.jsonl


ru/train-00000-of-00001.parquet:   0%|          | 0.00/6.50k [00:00<?, ?B/s]

ru/test-00000-of-00001.parquet:   0%|          | 0.00/56.5k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8 [00:00<?, ? examples/s]

...

en/test-00000-of-00001.parquet:   0%|          | 0.00/39.1k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/250 [00:00<?, ? examples/s]

mgsm/english:   0%|          | 0/20 [00:00<?, ?it/s]

mgsm/english: accuracy = 0.000 (0/20)
Saved: /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/results/base_qwen/mgsm_english_base_qwen.jsonl
